# Retrieval quality — Phase 1b measurement

Interactive companion to `tests/test_end_to_end.py`. Loads the 20 hand-labeled
conversations in `data/sample_conversations.jsonl` into a `HippocampalStore`,
runs the full retrieval pipeline (rule-based query planner → graph traversal →
rank) over a set of probe queries, and records per-query precision/recall plus
means.

Offline-friendly: uses `BonsaiQueryPlanner.plan_rule_based` so no Bonsai server
is required. The live Bonsai planner path is exercised by the live-gated tests
via the SSH tunnel; this notebook measures the deterministic retrieval engine.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

# Resolve the repo root from this notebook's location (notebooks/) and put
# it on sys.path so `from src...` imports work when the kernel starts in
# notebooks/.
_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.memory.episode import Episode
from src.memory.store import HippocampalStore
from src.retrieval.query_planner import BonsaiQueryPlanner
from src.retrieval.retriever import HippocampalRetriever

CORPUS = _ROOT / "data" / "sample_conversations.jsonl"

In [ ]:
# Build Episodes from the labeled corpus, chained with follows so the graph
# mirrors a real conversation history (same construction as the e2e tests).
rows = [json.loads(l) for l in CORPUS.read_text(encoding="utf-8").splitlines() if l.strip()]
episodes, prev = [], None
for i, r in enumerate(rows):
    turns = r.get("turns") or []
    u, a = turns[-1] if turns else ("", "")
    summary = a[:200] + ("..." if len(a) > 200 else "")
    episodes.append(Episode(
        id=r["id"],
        timestamp=f"2026-07-{(i % 28) + 1:02d}T10:00:00",
        summary=summary,
        full_text=f"User: {u}\nAssistant: {a}",
        entities=r.get("expected_entities") or [],
        topics=r.get("expected_topics") or [],
        tones=r.get("expected_tones") or [],
        decisions=r.get("expected_decisions") or [],
        follows=prev,
    ))
    prev = r["id"]

_TMP = tempfile.mkdtemp(prefix="hippo_retrieval_")
store = HippocampalStore(str(Path(_TMP) / "db"))
for ep in episodes:
    store.encode_episode(ep)

# Rule-based planner so the notebook runs offline (no Bonsai server).
class _RulePlanner:
    def __init__(self):
        self._p = BonsaiQueryPlanner()
    def plan(self, prompt):
        return self._p.plan_rule_based(prompt)

retriever = HippocampalRetriever(store, planner=_RulePlanner())
print(f"Loaded {len(episodes)} episodes into {Path(_TMP) / 'db'}")

In [ ]:
# Probe queries with expected answer sets (from the hand-labeled corpus).
# Expected sets are the ground truth the retrieval engine should surface.
_FRUSTRATED = {"conv_002", "conv_006", "conv_010", "conv_011", "conv_013",
               "conv_015", "conv_017", "conv_018", "conv_020"}

PROBES = [
    ("What was I frustrated about?", _FRUSTRATED),
    ("What did Alice and I decide?", {"conv_012", "conv_017"}),
]

def _precision_recall(retrieved_ids, expected):
    if not retrieved_ids and not expected:
        return 1.0, 1.0
    if not retrieved_ids:
        return 0.0, 0.0
    hit = len(set(retrieved_ids) & expected)
    precision = hit / len(set(retrieved_ids))
    recall = hit / len(expected) if expected else 1.0
    return precision, recall

rows_out = []
for query, expected in PROBES:
    results = retriever.retrieve(query)
    ids = [r["episode_id"] for r in results]
    p, rcl = _precision_recall(ids, expected)
    rows_out.append({"query": query, "retrieved": ids, "expected_n": len(expected),
                     "precision": p, "recall": rcl})
    print(f"Q: {query}")
    print(f"   retrieved={ids}")
    print(f"   precision={p:.2f}  recall={rcl:.2f}")
    print()

In [ ]:
import statistics

mean_p = statistics.mean(row["precision"] for row in rows_out)
mean_r = statistics.mean(row["recall"] for row in rows_out)
print(f"MEAN: precision={mean_p:.2f}, recall={mean_r:.2f}")
print()
print("Phase 1b retrieval checkpoint (graph-traversal engine, rule-based planner):")
print(f"  mean precision: {mean_p:.2f}  (target: 1.00 — no off-target episodes)")
print(f"  mean recall:    {mean_r:.2f}  (informational — recall depends on how")
print("                                  many labeled matches the query axis can reach)")
store.close()

## Notes

- **Precision is the gate, recall is informational.** The graph-traversal engine
  retrieves episodes that match the planned axis (tone / entity+topic / temporal
  chain). Precision should be 1.00 — every retrieved episode should be on-target.
  Recall reflects how many of the labeled matches the query's axis can reach
  (e.g., the frustrated-tone axis reaches all frustrated episodes; an entity+
  decision intersection reaches exactly the two Alice+decide convs).
- **Rule-based planner.** This notebook uses `plan_rule_based` so it runs
  without the Bonsai server. The live Bonsai planner is exercised by the
  live-gated tests (`tests/test_end_to_end.py`, `tests/test_mode_a.py`) via the
  SSH tunnel to the pod.
- **Phase F vector search** is not measured here — it's a semantic fallback
  that only fires when graph traversal returns <3 results. On the labeled
  corpus the graph axis is precise, so the fallback rarely triggers. Vector
  search quality is measured on the larger ingested corpus (Phase E) on the pod.